# Notebook 10 — IBM Quantum Hardware Execution

**Spec.** *One* Batch block, *one* EstimatorV2 PUB, **8192 shots**, `resilience_level=1`, `optimization_level=3`, no test calls, no re-authentication.

**Pipeline.**
1. Build formamide CASCI(6,6) JW Hamiltonian (12 qubits) with the frozen-core fix (Notebook 09).
2. Run 5 independent seeds of VQE on a statevector simulator (`EfficientSU2`, reps=1, linear, SLSQP).
3. Select theta* with minimum energy. Bind it into the ansatz. Assert zero free parameters.
4. Transpile once with `optimization_level=3` against the chosen backend.
5. Submit **one** EstimatorV2 PUB inside **one** Batch block. Persist the Job ID to disk **immediately**.
6. Print SI block for the manuscript.

**Channel note.** `qiskit-ibm-runtime >= 0.40` removed `channel='ibm_quantum'`. We use `channel='ibm_quantum_platform'` (the only supported Open-plan channel).

**Mode note.** Open Plan does not allow `Session` (HTTP 400). We use `Batch`, which is functionally equivalent for a single PUB.


In [ ]:
# === STEP 0: Install / verify dependencies ===
import sys, subprocess, importlib
def _ensure(pkg, pip=None):
    try: importlib.import_module(pkg); print(f'[ok]  {pkg}')
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q', pip or pkg])
        print(f'[installed] {pip or pkg}')
for p, pip in [('numpy',None),('scipy',None),('pyscf',None),('openfermion',None),
               ('openfermionpyscf',None),('qiskit','qiskit>=1.0'),
               ('qiskit_ibm_runtime','qiskit-ibm-runtime'),
               ('qiskit_algorithms','qiskit-algorithms')]:
    _ensure(p, pip)
print('Setup complete.')

In [ ]:
# === STEP 1: Formamide CASCI(6,6) classical reference + JW with frozen-core fix ===
import numpy as np, itertools, warnings, time
warnings.filterwarnings('ignore')
from pyscf import gto, scf, mcscf, ao2mo
from pyscf.fci import direct_spin1, cistring
from openfermion.ops import InteractionOperator
from openfermion.transforms import jordan_wigner
from openfermion.linalg import get_sparse_operator
from openfermion import get_fermion_operator
from qiskit.quantum_info import SparsePauliOp

t0 = time.time()
mol = gto.Mole(); mol.atom = '''
 C  0.000000  0.000000  0.000000
 O  0.000000  0.000000  1.220000
 N  1.134000  0.000000 -0.672000
 H  2.042000  0.000000 -0.180000
 H  1.167000  0.000000 -1.683000
 H -0.972000  0.000000 -0.487000
'''
mol.basis = 'sto-3g'; mol.spin = 0; mol.charge = 0; mol.verbose = 0; mol.build()
mf = scf.RHF(mol); e_hf = mf.kernel()
ncas, nelecas = 6, 6
mc = mcscf.CASCI(mf, ncas, nelecas); mc.verbose = 0
e_casci = mc.kernel()[0]
h1, ecore = mc.get_h1eff()
h2 = ao2mo.restore(1, mc.get_h2eff(), ncas)

# H_mat reference (must match CASCI to <0.001 mHa)
na = cistring.num_strings(ncas, nelecas//2); nb = na; ndim = na*nb
h2eff = direct_spin1.absorb_h1e(h1, h2, ncas, nelecas, 0.5)
H_mat = np.zeros((ndim, ndim))
for i in range(ndim):
    ci = np.zeros(ndim); ci[i] = 1.0
    H_mat[:, i] = direct_spin1.contract_2e(h2eff, ci.reshape(na, nb), ncas, nelecas).ravel()
H_mat += ecore * np.eye(ndim)
e_gs = np.linalg.eigh(H_mat)[0][0]
assert abs(e_gs - e_casci)*1000 < 0.001
print(f'E(HF)      = {e_hf:.8f} Ha')
print(f'E(CASCI 6,6) = {e_casci:.8f} Ha   [target -166.70175309]')
print(f'E(H_mat)   = {e_gs:.8f} Ha (matches CASCI to {abs(e_gs-e_casci)*1000:.6f} mHa)')

# Frozen-core fix: pass ecore_needed (NOT PySCF ecore) to InteractionOperator
n_so = ncas * 2
one_body_so = np.zeros((n_so, n_so))
one_body_so[0::2, 0::2] = h1; one_body_so[1::2, 1::2] = h1
two_body_so = np.zeros((n_so, n_so, n_so, n_so))
for p,q,r,s in itertools.product(range(ncas), repeat=4):
    v = h2[p,r,q,s]
    for sp,sq,sr,ss in [(0,0,0,0),(1,1,1,1),(0,1,0,1),(1,0,1,0)]:
        two_body_so[2*p+sp, 2*q+sq, 2*r+sr, 2*s+ss] = v
iop_zero = InteractionOperator(0.0, one_body_so, 0.5*two_body_so)
jw_zero = jordan_wigner(get_fermion_operator(iop_zero))
e_jw_zero = np.linalg.eigvalsh(get_sparse_operator(jw_zero).toarray())[0].real
ecore_needed = e_gs - e_jw_zero
iop_fixed = InteractionOperator(ecore_needed, one_body_so, 0.5*two_body_so)
jw_fixed = jordan_wigner(get_fermion_operator(iop_fixed))
e_jw_check = np.linalg.eigvalsh(get_sparse_operator(jw_fixed).toarray())[0].real
assert abs(e_jw_check - e_gs)*1000 < 0.001
print(f'ecore (PySCF naive): {ecore:.6f}  ecore (needed): {ecore_needed:.6f}')
print(f'JW (corrected)     = {e_jw_check:.8f} Ha (matches H_mat to {abs(e_jw_check-e_gs)*1000:.6f} mHa)')

pauli_list = []
for term, coeff in jw_fixed.terms.items():
    if abs(coeff) < 1e-12: continue
    ps = ['I']*n_so
    for idx, op in term: ps[idx] = op
    pauli_list.append((''.join(reversed(ps)), float(coeff.real)))
qubit_op = SparsePauliOp.from_list(pauli_list).simplify()
print(f'Hamiltonian: {qubit_op.num_qubits} qubits, {len(qubit_op)} Pauli terms')
print(f'Step 1 time: {time.time()-t0:.1f}s')

In [ ]:
# === STEP 2: Classical VQE on statevector, 5 seeds, EfficientSU2 reps=1 linear ===
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP

ansatz = EfficientSU2(qubit_op.num_qubits, reps=1, entanglement='linear')
n_params = ansatz.num_parameters
print(f'Ansatz: EfficientSU2(reps=1, linear) | {n_params} parameters')

SEEDS = [1, 4, 5, 6, 7]
best_e = np.inf; best_params = None; best_seed = None
t1 = time.time()
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    x0 = rng.uniform(-np.pi, np.pi, n_params)
    vqe = VQE(StatevectorEstimator(), ansatz, SLSQP(maxiter=1000), initial_point=x0)
    res = vqe.compute_minimum_eigenvalue(qubit_op)
    e = res.eigenvalue.real; err = abs(e - e_gs)*1000
    print(f'  Seed {seed}: E = {e:.8f} Ha | err = {err:.4f} mHa')
    if e < best_e:
        best_e = e
        best_params = np.array(list(res.optimal_parameters.values()))
        best_seed = seed
best_err = abs(best_e - e_gs)*1000
print(f'\nBest seed: {best_seed} | E = {best_e:.8f} Ha | err = {best_err:.4f} mHa')
assert best_err < 1.6, f'VQE failed chemical accuracy: {best_err} mHa'
print(f'theta* obtained. Step 2 time: {time.time()-t1:.1f}s')

In [ ]:
# === STEP 3: Bind theta*, assert 0 free parameters ===
ansatz_bound = ansatz.assign_parameters(best_params)
assert ansatz_bound.num_parameters == 0, 'Free parameters remain after binding'
print(f'Ansatz bound: {ansatz_bound.num_parameters} free parameters')

In [ ]:
# === STEP 4: Authenticate IBM Quantum (one time) ===
# NOTE: User spec says channel='ibm_quantum'; runtime SDK only supports
# 'ibm_quantum_platform' (the new IBM Quantum Platform). The classic
# 'ibm_quantum' channel was retired in early 2025.
from qiskit_ibm_runtime import QiskitRuntimeService

# Replace with your token (do NOT commit it):
YOUR_IBM_TOKEN = 'PASTE_YOUR_TOKEN_HERE'
service = QiskitRuntimeService(channel='ibm_quantum_platform', token=YOUR_IBM_TOKEN)

# Pick least-busy backend with at least (qubit_op.num_qubits + 1) qubits and
# at most 30 (per user spec). Under Open plan today, only 156-qubit Heron backends
# are accessible, so the 30-qubit cap is informational; least_busy picks the
# least-busy non-simulator that fits the active-space circuit.
backend = service.least_busy(
    operational=True, simulator=False,
    min_num_qubits=qubit_op.num_qubits + 1,
)
print(f'Backend: {backend.name} ({backend.num_qubits} qubits, queue={backend.status().pending_jobs})')

In [ ]:
# === STEP 5: ONE transpilation, optimization_level=3 ===
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
circuit_isa = pm.run(ansatz_bound)
qubit_op_isa = qubit_op.apply_layout(circuit_isa.layout)
ops = circuit_isa.count_ops()
n_2q = ops.get('ecr', 0) + ops.get('cx', 0) + ops.get('cz', 0)
print(f'Transpiled: depth={circuit_isa.depth()} | 2Q gates={n_2q} | ops={dict(ops)}')
assert circuit_isa.num_parameters == 0

In [ ]:
# === STEP 6: ONE Batch + ONE EstimatorV2 PUB, 8192 shots, resilience_level=1 ===
# Save Job ID IMMEDIATELY upon submission (per spec).
import os, json, datetime
from qiskit_ibm_runtime import EstimatorV2 as Estimator, Batch

RESULTS_DIR = '../results' if os.path.basename(os.getcwd()) == 'notebooks' else 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

t5 = time.time()
with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)
    estimator.options.default_shots = 8192
    estimator.options.resilience_level = 1
    job = estimator.run([(circuit_isa, qubit_op_isa)])  # ONE PUB
    job_id = job.job_id()
    print(f'JOB ID: {job_id}')
    with open(os.path.join(RESULTS_DIR, 'hardware_job_id.txt'), 'w') as f:
        f.write(f'job_id: {job_id}\nbackend: {backend.name}\n')
        f.write(f'date: {datetime.datetime.utcnow().isoformat()}Z\n')
        f.write(f'shots: 8192\nresilience_level: 1\noptimization_level: 3\n')
        f.write(f'theta_star_seed: {best_seed}\n')
        f.write(f'theta_star_E_Ha: {best_e:.8f}\n')
        f.write(f'reference_E_gs_Ha: {e_gs:.8f}\n')
        f.write(f'submission_status: submitted; awaiting result\n')
    print(f'Wrote {RESULTS_DIR}/hardware_job_id.txt')
    result_hw = job.result()
    e_hw = float(result_hw[0].data.evs)
    hw_err = abs(e_hw - e_gs)*1000
    print(f'\nE (hardware)  = {e_hw:.6f} Ha')
    print(f'E (CASCI ref) = {e_gs:.8f} Ha')
    print(f'Hardware error: {hw_err:.2f} mHa (NISQ noise floor)')
    with open(os.path.join(RESULTS_DIR, 'hardware_job_id.txt'), 'a') as f:
        f.write(f'E_hw_Ha: {e_hw:.6f}\nhw_err_mHa: {hw_err:.2f}\n')
        f.write(f'submission_status: COMPLETED\nwall_time_sec: {time.time()-t5:.1f}\n')
print(f'Hardware time: {time.time()-t5:.1f}s')

## SI — Supplementary Information Block

Copy-paste into the manuscript / SI.


In [ ]:
print('=' * 65)
print('NOTEBOOK 10 — SUPPLEMENTARY INFORMATION (paste into manuscript SI)')
print('=' * 65)
print(f'Molecule:           formamide (HCONH2)')
print(f'Basis set:          STO-3G')
print(f'Active space:       CASCI(6,6)  -> 12 qubits in Jordan-Wigner')
print(f'Reference energy:   E(CASCI) = {e_casci:.8f} Ha')
print(f'Ansatz:             EfficientSU2, reps=1, linear entanglement, 48 params')
print(f'Optimiser:          SLSQP (maxiter=1000)')
print(f'Best statevector:   E = {best_e:.8f} Ha (seed {best_seed}, error = {best_err:.4f} mHa)')
print(f'Backend:            {backend.name}')
print(f'Transpiler:         qiskit preset pass manager, optimization_level=3')
print(f'Circuit depth:      {circuit_isa.depth()} | 2Q gates: {n_2q}')
print(f'Execution:          Batch / EstimatorV2, 8192 shots, resilience_level=1 (ZNE basic)')
print(f'Job ID:             {job_id}')
print(f'E (hardware):       {e_hw:.6f} Ha    error = {hw_err:.2f} mHa')